In [ ]:
import os
from dotenv import load_dotenv
from reanimator.core import Reanimator
from reanimator.labelers import OpenAILabeler, LocalModelLabeler, TopicChunkPair, calculate_cohens_kappa
from reanimator.retrieval import Indexer, Retriever, reciprocal_rank_fusion, run_experiment
from reanimator.models import save_judgements, load_judgements

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
import pyterrier as pt
import nltk

load_dotenv()

import nltk
nltk.download('punkt_tab')

In [ ]:
reanimator = Reanimator(
    irds_name="irds:cord19/trec-covid",
    email="dummy@gmail.com",
    config={
        "downloader": {
            "email": "dummy@gmail.com"
        }
    }
)

In [3]:
#filter docs with original rel judgments to decrease candidate pool for faster processing in tutorial
# for testing take only 40 docs and topic 42
only_k = 40
topic_id = "42"

human_judgements = reanimator.source.get_qrels()
topics = reanimator.source.get_topics()
topic = [t for t in topics if t.query_id == topic_id][0]

doc_ids = [judg.doc_id for judg in human_judgements if judg.query_id == topic_id]
len(doc_ids)

There are multiple query fields available: ('title', 'description', 'narrative'). To use with pyterrier, provide variant or modify dataframe to add query column.


769

In [ ]:
docs = reanimator.load_documents(doc_ids=doc_ids)[:only_k]
reanimator.download_documents(docs)

#set accelerator options, device could be MPS, CUDA, CPU
accelerator_options = AcceleratorOptions(
        num_threads=8, device=AcceleratorDevice.MPS
    )

In [ ]:
# loading documents
docs = reanimator.load_documents(doc_ids=doc_ids)[:only_k]
reanimator.download_documents(docs)

In [ ]:
# extracting information with docling pipeline
reanimator.extract_content(docs)
reanimator.save_documents(docs, "documents")

In [ ]:
loaded_docs = reanimator.load_documents_from_file("documents")

In [9]:
#set different chunk modality types
chunks = reanimator.chunker.chunk(docs, metadata_fields_to_chunk=["title"])
table_chunks = [c for c in chunks if c.modality == "table"]
text_chunks = [c for c in chunks if c.modality == "text"]

print(f"{len(table_chunks)} table chunks")
print(f"{len(text_chunks)} text chunks")

263 table chunks
19954 text chunks


In [10]:
#generate indices for different chunk modalities

indexer_both = Indexer(index_type="bm25", path="data/indices/bm25_both/")
indexer_table = Indexer(index_type="bm25", path="data/indices/bm25_table/")
indexer_text = Indexer(index_type="bm25", path="data/indices/bm25_text/")

indexer_both.index(chunks)
indexer_table.index(table_chunks)
indexer_text.index(text_chunks)

Creating new indexes...
Indexes created and saved.
Creating new indexes...
Indexes created and saved.
Creating new indexes...
Indexes created and saved.


In [11]:
retriever_both = Retriever(indexer=indexer_both)
retriever_table = Retriever(indexer=indexer_table)
retriever_text = Retriever(indexer=indexer_text)

In [15]:
#generate pool of chunks from different retrieval systems for synthetic rel judgements
res_both = retriever_both.retrieve(topic=topic, k=20)
res_table = retriever_table.retrieve(topic=topic, k=20)
res_text = retriever_text.retrieve(topic=topic, k=20)

pool = set(reciprocal_rank_fusion([res_both, res_table, res_text]))
len(pool)

42

## Synthetic Relevance Judgments

In [12]:
# using local model
labeler_qwen = LocalModelLabeler(model="qwen/qwen3-30b-a3b", 
                            base_url="http://192.168.178.180:1234/v1", 
                            concurrency=10,
                            thinking=False)

INFO: LocalModelLabeler initialized with model: qwen/qwen3-30b-a3b at http://192.168.178.180:1234/v1


In [13]:
# using openai model
labeler_gpt41mini = OpenAILabeler(api_key=os.getenv("OPENAI_API_KEY"), prompt_path="../src/reanimator/default_prompt.txt")
model = labeler_gpt41mini.model

INFO: OpenAILabeler initialized with model: gpt-4.1-mini-2025-04-14


In [16]:
#set batch to for labeling which appear in the pool
batch = [TopicChunkPair(topic=topic, chunk=chunk) for chunk in chunks if chunk.chunk_id in pool]
len(batch)

42

In [ ]:
#generate local model synthetic rel judgements

qwen_judgements = await labeler_qwen.label_all(batch)

model_name = labeler_qwen.model.replace("/", "_")
save_judgements(qwen_judgements, f"data/judgments/machine_{model_name}_judgements.json")

In [18]:
#generate openai model synthetic rel judgements
gpt_judgements = await labeler_gpt41mini.label_all(batch)

model_name = labeler_gpt41mini.model.replace("/", "_")
save_judgements(gpt_judgements, f"data/judgments/machine_{model_name}_judgements.json")

Generating Judgements: 100%|██████████| 42/42 [00:05<00:00,  7.68it/s]


In [19]:
from reanimator.models import save_judgements
model = model.replace("/", "_")
save_judgements(gpt_judgements, f"machine_{model}_judgements.json")

## Human Relevance Judgments


In [20]:
from reanimator.labeling import *

In [32]:
# Load machine labeled pairs here or use judgments from previous steps:
machine_judgements = [{"chunk_id":a.chunk_id, "query_id":a.query_id, "doc_id":a.doc_id, "source":a.source} for a in gpt_judgements]

In [34]:
# insert path to your labeling log file 
output_path = "data/judgments/human1_judgements.json"
# what modality are you labeling: text or table
modality = "text"

In [35]:
label_chunks = [a.to_dict() for a in chunks]
label_topics = [t.to_dict() for t in topics]

In [37]:
to_label = load_label_pairs(machine_judgements, label_chunks, output_path=output_path, modality=modality, num_pairs=20)

20  texts left to label: 20 pairs to label, 0 already labeled in 'data/judgments/human1_judgements.json'.


In [38]:
labeling_interface(output_path=output_path, sampled=to_label, topics=label_topics, chunks=label_chunks)

RadioButtons(description='Relevance:', layout=Layout(width='50%'), options=(('0: Not relevant', 0), ('1: Parti…

Button(button_style='primary', description='Submit', style=ButtonStyle())

Output()

RadioButtons(description='Relevance:', layout=Layout(width='50%'), options=(('0: Not relevant', 0), ('1: Parti…

Button(button_style='primary', description='Submit', style=ButtonStyle())

...

Button(button_style='primary', description='Submit', style=ButtonStyle())

Output()

RadioButtons(description='Relevance:', layout=Layout(width='50%'), options=(('0: Not relevant', 0), ('1: Parti…

Button(button_style='primary', description='Submit', style=ButtonStyle())

Output()

## Label Evaluation

In [ ]:
#calculate cohens kappa between two labelers

kappa = calculate_cohens_kappa("data/judgments/machine_gpt-4.1-mini-2025-04-14_judgements.json", "data/judgments/machine_qwen_qwen3-30b-a3b_judgements.json")

In [ ]:
qwen_judgements = load_judgements("data/judgments/machine_qwen_qwen3-30b-a3b_judgements.json")

In [ ]:
run_experiment(rankings=[res_both, res_table, res_text], topics=topics, judgements=qwen_judgements, eval_metrics=[pt.measures.nDCG, pt.measures.P@20], names=["BM25", "BM25 Table", "BM25 Text"])